## 2026-27 Live Title Simulation

---
## Step 1 - Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import soccerdata as sd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import log_loss, accuracy_score

warnings.filterwarnings('ignore')
%matplotlib inline

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

TITLE_TEAMS = ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United']
BIG6 = ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United', 'Chelsea', 'Tottenham']

RIVALRIES = {
    'Arsenal': ['Tottenham', 'Manchester United', 'Chelsea', 'Manchester City'],
    'Brentford': ['Fulham', 'Chelsea'],
    'Everton': ['Liverpool'],
    'Hull': ['Leeds'],
    'Ipswich': [],  # Main rival (Norwich City) is not in the Premier League this season
    'Nottingham Forest': ['Coventry', 'Aston Villa'],
    'Brighton': ['Crystal Palace', 'Bournemouth'],
    'Manchester City': ['Manchester United', 'Liverpool', 'Arsenal'],
    'Newcastle United': ['Sunderland'],
    'Fulham': ['Chelsea', 'Brentford'],
    'Crystal Palace': ['Brighton'],
    'Bournemouth': ['Brighton'],
    'Coventry': ['Aston Villa', 'Nottingham Forest'],
    'Liverpool': ['Manchester United', 'Everton', 'Manchester City', 'Chelsea'],
    'Tottenham': ['Arsenal', 'Chelsea'],
    'Chelsea': ['Arsenal', 'Tottenham', 'Fulham', 'Leeds', 'Liverpool'],
    'Leeds': ['Manchester United', 'Chelsea', 'Hull'],
    'Manchester United': ['Liverpool', 'Manchester City', 'Leeds', 'Arsenal'],
    'Sunderland': ['Newcastle United'],
    'Aston Villa': ['Coventry', 'Nottingham Forest']
}

ARTETA_SEASONS = ['1920', '2021', '2122', '2223', '2324', '2425', '2526']

TEAM_PALETTE = {
    'Arsenal':            '#EF0107',
    'Liverpool':          '#00B2A9',
    'Manchester City':    '#6CABDD',
    'Manchester United':  '#FFB81C',
}

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

print('Imports ready')

[09/18/26 16:26:40] INFO     No custom team name replacements found. You can configure these in       ]8;id=403823;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=403824;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py#91\91]8;;\
                             C:\Users\tejas\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=403830;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=403831;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py#189\189]8;;\
                             C:\Users\tejas\soccerdata\config\league_dict.json.                                    

Imports ready


---
## Step 2 - Load Data

In [2]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PROC_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

df = pd.read_csv(PROC_DATA_DIR / 'all_4teams_processed.csv')
df['date'] = pd.to_datetime(df['date'])
df['season'] = df['season'].astype(str)

print(f'Shape: {df.shape}')

Shape: (1064, 37)


---
## Section A - What a Monte Carlo Simulation Actually Is

A quick illustration of drawing random outcomes from a fixed probability triple, confirming the sampling mechanic behaves as expected before it gets used for real.

In [3]:
# a quick illustration: one match, simulated 10,000 times, using its own random draw each time
p_win, p_draw, p_loss = 0.60, 0.25, 0.15

outcomes = rng.choice(['Win', 'Draw', 'Loss'], size=10000, p=[p_win, p_draw, p_loss])
observed = pd.Series(outcomes).value_counts(normalize=True).reindex(['Win', 'Draw', 'Loss'])

print('Input probabilities:  Win 0.60, Draw 0.25, Loss 0.15')
print('Observed frequency across 10,000 independent draws:')
print(observed.round(3))

Input probabilities:  Win 0.60, Draw 0.25, Loss 0.15
Observed frequency across 10,000 independent draws:
Win     0.605
Draw    0.248
Loss    0.148
Name: proportion, dtype: float64


---
## Section B - The Live Data Problem

NB01's `fixtures_26-27.csv` / `all_teams_26-27_raw.csv` were scraped from FBRef with `xG`/`xGA` left blank and may be several gameweeks stale. Check their actual state directly rather than assuming a file called "raw" is current.

In [4]:
fx_old = pd.read_csv(RAW_DATA_DIR / 'all_teams_26-27_raw.csv')
print(f'Rows: {len(fx_old)}, played (non-null scored): {fx_old["scored"].notna().sum()}')
print(f'xG non-null count: {fx_old["xG"].notna().sum()}')
print(f'Max gameweek with a real result: {fx_old.loc[fx_old["scored"].notna(), "gameweek"].max()}')

Rows: 760, played (non-null scored): 20
xG non-null count: 0
Max gameweek with a real result: 1


In [5]:
u_check = sd.Understat(leagues='ENG-Premier League', seasons='2627', no_cache=True)
sched_check = u_check.read_schedule().reset_index()
sched_check['date'] = pd.to_datetime(sched_check['date'])
played = sched_check[sched_check['home_goals'].notna()]

print(f'Understat 2026-27 matches with a real result: {len(played)}')
print(f'Latest played date: {played["date"].max()}')
print()
print('A sample played match, home_xg/away_xg populated:')
print(played[['date','home_team','away_team','home_goals','away_goals','home_xg','away_xg']].tail(3).to_string(index=False))

[09/18/26 16:26:42] INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=403838;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=403839;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-09-18 16:26:42] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Successfully loaded TLS library:                                      ]8;id=403846;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=403847;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\                 
                             bin\tls-client-xgo-1.13.1-windows-amd64.dll                                           

Understat 2026-27 matches with a real result: 40
Latest played date: 2026-09-14 19:00:00

A sample played match, home_xg/away_xg populated:
               date         home_team        away_team  home_goals  away_goals   home_xg   away_xg
2026-09-13 13:00:00          Coventry         Brighton           0           5   1.04812   2.76518
2026-09-13 15:30:00 Manchester United  Manchester City           0           1  0.943406   1.14073
2026-09-14 19:00:00             Leeds Newcastle United           4           1    2.2902  0.793165


---
## Section C - Building the Live Update

Pull the current 20-team schedule fresh from Understat (`no_cache=True`), reshape it into this project's usual team-per-row long format, and write it to new `fixtures_26-27_live.csv` / `all_teams_26-27_live.csv` files, leaving NB01's original output untouched.

In [6]:
def fetch_live_schedule(season='2627'):
    u = sd.Understat(leagues='ENG-Premier League', seasons = season, no_cache=True)
    sched = u.read_schedule().reset_index()
    sched['date'] = pd.to_datetime(sched['date'])

    home = sched.rename(columns={
        'home_team': 'team', 'away_team': 'opponent',
        'home_goals': 'scored', 'away_goals': 'conceded',
        'home_xg': 'xG', 'away_xg': 'xGA',        
    })
    home['venue'] = 'home'
    away = sched.rename(columns={
        'away_team': 'team', 'home_team': 'opponent',
        'away_goals': 'scored', 'home_goals': 'conceded',
        'away_xg': 'xG', 'home_xg': 'xGA',
    })
    away['venue'] = 'away'

    cols = ['league', 'season', 'game_id', 'date', 'team', 'opponent', 'venue', 'xG', 'xGA', 'scored', 'conceded']
    long = pd.concat([home[cols], away[cols]], ignore_index=True)
    long['gameweek'] = long.groupby('team')['date'].rank(method='dense').astype(int)
    return long.sort_values(['team','date']).reset_index(drop=True)

live_2627 = fetch_live_schedule()
print(f'Live 2026-27 long format: {live_2627.shape}')
print(f'Teams: {live_2627["team"].nunique()}')
print(f'Played rows: {live_2627["scored"].notna().sum()} / {len(live_2627)}')
print(f'Latest played gameweek: {live_2627.loc[live_2627["scored"].notna(), "gameweek"].max()}')

[09/18/26 16:26:45] INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=403852;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=403853;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

Live 2026-27 long format: (760, 12)
Teams: 20
Played rows: 80 / 760
Latest played gameweek: 4


In [7]:
all20_live = live_2627.copy()
tracked_live = live_2627[live_2627['team'].isin(TITLE_TEAMS)].copy()

all20_live.to_csv(RAW_DATA_DIR / 'all_teams_26-27_live.csv', index=False)
tracked_live.to_csv(RAW_DATA_DIR / 'fixtures_26-27_live.csv', index=False)

print(f'Wrote all_teams_26-27_live.csv : {len(all20_live)} rows')
print(f'Wrote fixtures_26-27_live.csv  : {len(tracked_live)} rows')

Wrote all_teams_26-27_live.csv : 760 rows
Wrote fixtures_26-27_live.csv  : 152 rows


---
## Section D - Scope: One Model, Not Two

A genuine title race needs all 20 teams, not just the 4 tracked here. Training a second, simpler model for the other 16 looks reasonable, check whether it introduces a population bias before committing to it.

In [8]:
# rebuild the two-model version first, to see the problem directly
hist = sd.Understat(leagues='ENG-Premier League', seasons=ARTETA_SEASONS)
hist_sched = hist.read_schedule().reset_index()
hist_sched['date'] = pd.to_datetime(hist_sched['date'])

def wide_to_long(sched):
    home = sched.rename(columns={'home_team':'team','away_team':'opponent','home_goals':'scored',
                                   'away_goals':'conceded','home_xg':'xG','away_xg':'xGA'})
    home['venue'] = 'home'
    away = sched.rename(columns={'away_team':'team','home_team':'opponent','away_goals':'scored',
                                   'home_goals':'conceded','away_xg':'xG','home_xg':'xGA'})
    away['venue'] = 'away'
    cols = ['league','season','game_id','date','team','opponent','venue','xG','xGA','scored','conceded']
    return pd.concat([home[cols], away[cols]], ignore_index=True).sort_values(['team','season','date']).reset_index(drop=True)

full_long = wide_to_long(hist_sched)
for c in ['scored','conceded','xG','xGA']:
    full_long[c] = full_long[c].astype('float64')

full_long['result'] = np.where(full_long['scored'] > full_long['conceded'], 'W',
    np.where(full_long['scored'] == full_long['conceded'], 'D','L'))
full_long['points'] = full_long['result'].map({'W':3,'D':1,'L':0})
full_long['xgd'] = full_long['xG']- full_long['xGA']
full_long['is_win'] = (full_long['result'] == 'W').astype(float)
full_long['is_home'] = (full_long['venue'] == 'home').astype(int)
full_long['is_big6_opp'] = full_long['opponent'].isin(BIG6).astype(int)

def roll(g, col, window, min_p= 3):
    return g[col].transform(lambda x: x.shift(1).rolling(window, min_periods=min_p).mean())
g = full_long.groupby(['team','season'])
full_long['xG_roll5'] = roll(g, 'xG',5)
full_long['xGA_roll5'] = roll(g, 'xGA', 5)
full_long['pts_roll5'] = roll(g, 'points', 5)
full_long['win_rate_roll5'] = roll(g, 'is_win', 5)
full_long['xgd_roll5'] = roll(g, 'xgd', 5)

opp_lookup = full_long[['team','season','game_id','xgd_roll5']].rename(columns={'team':'opponent','xgd_roll5':'opp_xgd_roll5'})
full_long = full_long.merge(opp_lookup, on = ['opponent','season','game_id'], how = 'left')
full_long['parity_gap'] = (full_long['xgd_roll5'] - full_long['opp_xgd_roll5']).abs()

print(f'Full 20-team long shape: {full_long.shape}')

[09/18/26 16:26:48] INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=403858;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=403859;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

Full 20-team long shape: (5320, 24)


In [9]:
FEATURES_T2 = ['xG_roll5','xGA_roll5','pts_roll5','win_rate_roll5','is_home','is_big6_opp','opp_xgd_roll5','parity_gap']

all_rows = full_long[full_long['season'].isin(ARTETA_SEASONS)].copy()
y_map = {'L':0,'D':1,'W':2}
all_rows['y'] = all_rows['result'].map(y_map)
t2_df = all_rows.dropna(subset = FEATURES_T2 + ['y']).copy()
t2_df['season_int'] = t2_df['season'].astype(int)
t2_train, t2_test = t2_df[t2_df['season_int']<=2425] , t2_df[t2_df['season_int'] == 2526]

sc2 = StandardScaler()
m2 = LogisticRegression(max_iter=2000).fit(sc2.fit_transform(t2_train[FEATURES_T2]), t2_train['y'])

avg_t2 = t2_train[FEATURES_T2].mean().to_frame().T
p_avg_t2 = m2.predict_proba(sc2.transform(avg_t2))[0]
print(f'Tier-2 model (all 20 teams), average team baseline -> L/D/W = {p_avg_t2.round(3)}')

Tier-2 model (all 20 teams), average team baseline -> L/D/W = [0.375 0.25  0.375]


In [10]:
FEATURES_T1 = ['xG_roll5','xGA_roll5','pts_roll5','win_rate_roll5','stakes_intensity','is_home',
               'is_big6_opp','is_non_big6_rivalry','opp_xgd_roll5']

t1_processed = pd.read_csv(PROC_DATA_DIR / 'all_4teams_processed.csv')
t1_processed['is_home'] = (t1_processed['venue'] == 'home')
t1_processed['y'] = t1_processed['result'].map(y_map)

# is_big6_opp / is_non_big6_rivalry / opp_xgd_roll5 aren't saved in the processed CSV,
# NB06 computed them inline, so this rebuilds the same 3 columns the same way
t1_processed['is_big6_opp'] = t1_processed['opponent'].isin(BIG6).astype(int)
t1_processed['is_non_big6_rivalry'] = (t1_processed['is_rivalry'] & (t1_processed['is_big6_opp']==0)).astype(int)
opp_lookup_t1 = full_long[['team','season','game_id','xgd_roll5']].rename(columns={'team':'opponent','xgd_roll5':'opp_xgd_roll5'})
t1_processed['season'] = t1_processed['season'].astype(str)
opp_lookup_t1['season'] = opp_lookup_t1['season'].astype(str)
t1_processed = t1_processed.merge(opp_lookup_t1, on=['opponent','season','game_id'], how='left')

t1_df = t1_processed.dropna(subset=FEATURES_T1 + ['y']).copy()
t1_df['season_int'] = t1_df['season'].astype(int)
t1_train = t1_df[t1_df['season_int']<=2425]

sc1 = StandardScaler()
m1 = LogisticRegression(max_iter=2000).fit(sc1.fit_transform(t1_train[FEATURES_T1]), t1_train['y'])
avg_t1 = t1_train[FEATURES_T1].mean().to_frame().T
p_avg_t1 = m1.predict_proba(sc1.transform(avg_t1))[0]
print(f'Tier-1 model (4 tracked teams), average team baseline -> L/D/W = {p_avg_t1.round(3)}')
print(f'Tier-2 model (all 20 teams),   average team baseline -> L/D/W = {p_avg_t2.round(3)}')
print(f'Win probability gap from model choice alone: {p_avg_t1[2]-p_avg_t2[2]:+.3f}')

Tier-1 model (4 tracked teams), average team baseline -> L/D/W = [0.181 0.213 0.606]
Tier-2 model (all 20 teams),   average team baseline -> L/D/W = [0.375 0.25  0.375]
Win probability gap from model choice alone: +0.232


---
## Section E - Stakes Intensity for All 20 Teams

`title_gap`, `cl_gap`, and `eur_gap` already generalise to all 20 teams from NB02's full-schedule reconstruction. Extend `stakes_intensity` beyond the 4 tracked teams, and add the relegation boundary the original formula never needed.

In [13]:
# reconstruct the full stakes_intensity ingredients for ALL 20 teams
full_long['gameweek'] = full_long.groupby(['team','season'])['date'].rank(method='dense').astype(int)
full_long['cum_pts'] = full_long.groupby(['team','season'])['points'].transform(lambda x: x.fillna(0).cumsum())
full_long['leader_pts'] = full_long.groupby(['season','gameweek'])['cum_pts'].transform('max')
full_long['title_gap'] = full_long['leader_pts'] - full_long['cum_pts']

def nth_place_pts(x, n):
    return x.nlargest(n).iloc[-1] if len(x) >= n else x.min()

for n, name in [(4,'4th'), (5,'5th'), (6,'6th'), (7,'7th'), (17,'17th'), (18,'18th')]:
    full_long[f'pts_{name}'] = full_long.groupby(['season','gameweek'])['cum_pts'].transform(lambda x: nth_place_pts(x, n))

def boundary_gap(cum_pts, top, next_):
    return np.where(cum_pts >= top, top - next_, top - cum_pts)

full_long['cl_gap'] = boundary_gap(full_long['cum_pts'], full_long['pts_4th'], full_long['pts_5th'])
full_long['eur_gap'] = boundary_gap(full_long['cum_pts'], full_long['pts_6th'], full_long['pts_7th'])
# NOT boundary_gap here: that hands every safe team the same flat 17th-vs-18th cushion
# regardless of how far clear they are. Invisible for cl_gap/eur_gap since only 4-6 teams
# can ever be "safe" there and title_gap usually dominates anyway, but with up to 17 teams
# "safe" from relegation, it wrongly flagged mid-table teams with nothing else going on.
# Fixed by using each team's own distance above 18th place instead of the shared cushion.
full_long['releg_gap'] = np.where(
    full_long['cum_pts'] >= full_long['pts_17th'],
    full_long['cum_pts'] - full_long['pts_18th'],
    full_long['pts_17th'] - full_long['cum_pts']
)

GW38_MAX = 1 / (1 + np.exp(-0.2 * (38 - 22)))
gw_factor = 1 / (1 + np.exp(-0.2 * (full_long['gameweek'] - 22)))
pts_remaining = (38 - full_long['gameweek'] + 1) * 3

title_f = (1 - full_long['title_gap'] / pts_remaining).clip(lower=0)
cl_f = (1 - full_long['cl_gap'] / pts_remaining).clip(lower=0)
eur_f = (1 - full_long['eur_gap'] / pts_remaining).clip(lower=0)
releg_f = (1 - full_long['releg_gap'] / pts_remaining).clip(lower=0)

title_raw = gw_factor * title_f
cl_raw = gw_factor * cl_f * 0.75
eur_raw = gw_factor * eur_f * 0.50
releg_raw = gw_factor * releg_f * 0.80

base_stakes = np.maximum.reduce([title_raw, cl_raw, eur_raw, releg_raw]) / GW38_MAX

recent_form = full_long.groupby(['team','season'])['points'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()).fillna(1.5)
form_mult = np.where(recent_form < 1.0, 1.15, 1.0)
full_long['is_rivalry'] = full_long.apply(lambda r: r['opponent'] in RIVALRIES.get(r['team'], []), axis=1)
rivalry_bonus = np.where(full_long['is_rivalry'], 0.15, 0)

full_long['stakes_intensity'] = (base_stakes * form_mult + rivalry_bonus).clip(0, 1).round(4)
full_long['is_non_big6_rivalry'] = (full_long['is_rivalry'] & (full_long['is_big6_opp']==0)).astype(int)

print('stakes_intensity now computed for all 20 teams, not just the 4 tracked ones.')
print(full_long.groupby(full_long['team'].isin(TITLE_TEAMS))['stakes_intensity'].describe()[['mean','max']].round(3))

stakes_intensity now computed for all 20 teams, not just the 4 tracked ones.
        mean  max
team             
False  0.257  1.0
True   0.371  1.0


In [14]:
snap_35 = full_long[(full_long['season']=='2526') & (full_long['gameweek']==35)].sort_values('cum_pts', ascending=False)
print('Whole table, 2025-26 GW35:')
print(snap_35[['team','cum_pts','title_gap','releg_gap','stakes_intensity']].head(8).round(3).to_string(index=False))

Whole table, 2025-26 GW35:
             team  cum_pts  title_gap  releg_gap  stakes_intensity
          Arsenal       76          0         40             0.969
  Manchester City       74          2         38             0.807
Manchester United       64         12         28             0.877
      Aston Villa       58         18         22             0.727
        Liverpool       58         18         22             0.877
      Bournemouth       52         24         16             0.444
        Brentford       51         25         15             0.511
         Brighton       50         26         14             0.404


In [15]:
# fix: for a safe team, use ITS OWN distance above the drop zone, not the shared cushion
full_long['relag_gap'] = np.where(
    full_long['cum_pts'] >= full_long['pts_17th'],
    full_long['cum_pts'] - full_long['pts_18th'],
    full_long['pts_17th'] - full_long['cum_pts']
)

releg_f = (1 - full_long['releg_gap'] / pts_remaining).clip(lower=0)
releg_raw = gw_factor * releg_f * 0.80
base_stakes = np.maximum.reduce([title_raw, cl_raw, eur_raw, releg_raw]) / GW38_MAX
full_long['stakes_intensity'] = (base_stakes * form_mult + rivalry_bonus).clip(0, 1).round(4)

snap_35 = full_long[(full_long['season']=='2526') & (full_long['gameweek']==35)].sort_values('cum_pts', ascending=False)
print('Same gameweek, fixed releg_gap:')
print(snap_35[['team','cum_pts','releg_gap','stakes_intensity']].head(8).round(3).to_string(index=False))

Same gameweek, fixed releg_gap:
             team  cum_pts  releg_gap  stakes_intensity
          Arsenal       76         40             0.969
  Manchester City       74         38             0.807
Manchester United       64         28             0.877
      Aston Villa       58         22             0.727
        Liverpool       58         22             0.877
      Bournemouth       52         16             0.444
        Brentford       51         15             0.511
         Brighton       50         14             0.404



---
## Section F - One More Real Signal: Head-to-Head Record

Test head-to-head record against the specific opponent, and whatever else seems worth trying, against the same holdout, keeping only what actually survives.

In [18]:
all_rows = full_long[full_long['season'].isin(ARTETA_SEASONS)].copy()
all_rows['y'] = all_rows['result'].map(y_map)

RECENCY = {'1920':1.0,'2021':1.0,'2122':1.0,'2223':1.1,'2324':1.2,'2425':1.3,'2526':1.5}
all_rows['recency_weight'] = all_rows['season'].map(RECENCY)

h2h = all_rows[['team','opponent','date','points']].sort_values(['team','opponent','date']).copy()
h2h['h2h_pts_avg3'] = h2h.groupby(['team','opponent'])['points'].transform(lambda x: x.shift(1).rolling(3, min_periods = 1).mean())
all_rows = all_rows.merge(h2h[['team','opponent','date','h2h_pts_avg3']], on = ['team','opponent','date'], how = 'left')
all_rows['h2h_pts_avg3'] = all_rows['h2h_pts_avg3'].fillna(1.5)

FEATURES = ['xG_roll5','xGA_roll5','pts_roll5','win_rate_roll5','is_home','is_big6_opp',
            'opp_xgd_roll5','parity_gap','stakes_intensity','is_non_big6_rivalry']
FEATURES_H2H = FEATURES + ['h2h_pts_avg3']


model_df = all_rows.dropna(subset=FEATURES_H2H + ['y']).copy()
model_df['season_int'] = model_df['season'].astype(int)
train, test = model_df[model_df['season_int']<=2425], model_df[model_df['season_int']==2526]

def fit_eval(features):
    sc = StandardScaler()
    Xtr, Xte = sc.fit_transform(train[features]), sc.transform(test[features])
    m = LogisticRegression(max_iter=2000).fit(Xtr, train['y'], sample_weight = train['recency_weight'])
    p = m.predict_proba(Xte)
    return log_loss(test['y'], p, labels=[0,1,2]), accuracy_score(test['y'], m.predict(Xte)), m, sc

ll_without, acc_without, _, _ = fit_eval(FEATURES)
ll_with, acc_with, m_final, sc_final = fit_eval(FEATURES_H2H)

print(f'Without head-to-head: log loss {ll_without:.4f}, accuracy {acc_without:.4f}')
print(f'With head-to-head:    log loss {ll_with:.4f}, accuracy {acc_with:.4f}')

Without head-to-head: log loss 1.0455, accuracy 0.4757
With head-to-head:    log loss 1.0362, accuracy 0.4886


---
## Section G - The Production Model, and What "Today" Means

Refit the chosen recipe on everything through 2025-26 rather than holding it back as a permanent test set, then build each team's frozen form snapshot as of their next unplayed 2026-27 fixture.

In [19]:
# rebuild the whole feature pipeline once more, this time on historical + live combined,
# so the rolling features and stakes_intensity extend all the way through today's real matches
live_sched = sd.Understat(leagues='ENG-Premier League', seasons='2627', no_cache=True).read_schedule().reset_index()
live_sched['date'] = pd.to_datetime(live_sched['date'])
live_long = wide_to_long(live_sched)

combined = pd.concat([full_long[['league','season','game_id','date','team','opponent','venue','xG','xGA','scored','conceded']],
                       live_long], ignore_index=True)
for c in ['scored','conceded','xG','xGA']:
    combined[c] = combined[c].astype('float64')
combined = combined.sort_values(['team','season','date']).reset_index(drop=True)

combined['result'] = np.where(combined['scored']>combined['conceded'], 'W', np.where(combined['scored']==combined['conceded'], 'D', 'L'))
combined.loc[combined['scored'].isna(), 'result'] = np.nan
combined['points'] = combined['result'].map({'W':3,'D':1,'L':0})
combined['xgd'] = combined['xG'] - combined['xGA']
combined['is_win'] = (combined['result']=='W').astype(float)
combined['is_home'] = (combined['venue']=='home').astype(int)
combined['is_big6_opp'] = combined['opponent'].isin(BIG6).astype(int)
combined['is_rivalry'] = combined.apply(lambda r: r['opponent'] in RIVALRIES.get(r['team'], []), axis=1)
combined['is_non_big6_rivalry'] = (combined['is_rivalry'] & (combined['is_big6_opp']==0)).astype(int)

g = combined.groupby(['team','season'])
combined['xG_roll5'] = roll(g, 'xG', 5)
combined['xGA_roll5'] = roll(g, 'xGA', 5)
combined['pts_roll5'] = roll(g, 'points', 5)
combined['win_rate_roll5'] = roll(g, 'is_win', 5)
combined['xgd_roll5'] = roll(g, 'xgd', 5)

opp_lookup = combined[['team','season','game_id','xgd_roll5']].rename(columns={'team':'opponent','xgd_roll5':'opp_xgd_roll5'})
combined = combined.merge(opp_lookup, on=['opponent','season','game_id'], how='left')
combined['parity_gap'] = (combined['xgd_roll5'] - combined['opp_xgd_roll5']).abs()

combined['gameweek'] = combined.groupby(['team','season'])['date'].rank(method='dense').astype(int)
combined['cum_pts'] = combined.groupby(['team','season'])['points'].transform(lambda x: x.fillna(0).cumsum())
combined['leader_pts'] = combined.groupby(['season','gameweek'])['cum_pts'].transform('max')
combined['title_gap'] = combined['leader_pts'] - combined['cum_pts']
for n, name in [(4,'4th'), (5,'5th'), (6,'6th'), (7,'7th'), (17,'17th'), (18,'18th')]:
    combined[f'pts_{name}'] = combined.groupby(['season','gameweek'])['cum_pts'].transform(lambda x: nth_place_pts(x, n))
combined['cl_gap'] = boundary_gap(combined['cum_pts'], combined['pts_4th'], combined['pts_5th'])
combined['eur_gap'] = boundary_gap(combined['cum_pts'], combined['pts_6th'], combined['pts_7th'])
# fixed version from Section E: a safe team's own distance above 18th, not the shared cushion
combined['releg_gap'] = np.where(
    combined['cum_pts'] >= combined['pts_17th'],
    combined['cum_pts'] - combined['pts_18th'],
    combined['pts_17th'] - combined['cum_pts']
)

gw_factor_c = 1 / (1 + np.exp(-0.2 * (combined['gameweek'] - 22)))
pts_remaining_c = (38 - combined['gameweek'] + 1) * 3
title_f_c = (1 - combined['title_gap']/pts_remaining_c).clip(lower=0)
cl_f_c = (1 - combined['cl_gap']/pts_remaining_c).clip(lower=0)
eur_f_c = (1 - combined['eur_gap']/pts_remaining_c).clip(lower=0)
releg_f_c = (1 - combined['releg_gap']/pts_remaining_c).clip(lower=0)
base_stakes_c = np.maximum.reduce([gw_factor_c*title_f_c, gw_factor_c*cl_f_c*0.75, gw_factor_c*eur_f_c*0.50, gw_factor_c*releg_f_c*0.80]) / GW38_MAX
recent_form_c = combined.groupby(['team','season'])['points'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()).fillna(1.5)
form_mult_c = np.where(recent_form_c < 1.0, 1.15, 1.0)
rivalry_bonus_c = np.where(combined['is_rivalry'], 0.15, 0)
combined['stakes_intensity'] = (base_stakes_c*form_mult_c + rivalry_bonus_c).clip(0,1).round(4)

h2h_c = combined[['team','opponent','date','points']].sort_values(['team','opponent','date']).copy()
h2h_c['h2h_pts_avg3'] = h2h_c.groupby(['team','opponent'])['points'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
combined = combined.merge(h2h_c[['team','opponent','date','h2h_pts_avg3']], on=['team','opponent','date'], how='left')
combined['h2h_pts_avg3'] = combined['h2h_pts_avg3'].fillna(1.5)

RECENCY['2627'] = 1.6
combined['recency_weight'] = combined['season'].map(RECENCY)

print(f'Combined historical + live shape: {combined.shape}')
print(f'2026-27 rows: {(combined["season"]=="2627").sum()}, played: {((combined["season"]=="2627") & combined["scored"].notna()).sum()}')

[09/18/26 17:05:48] INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=403864;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=403865;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

Combined historical + live shape: (6080, 42)
2026-27 rows: 760, played: 80


In [21]:
FEATURES_FINAL = FEATURES_H2H
prod_df = combined[combined['season'].isin(ARTETA_SEASONS)].dropna(subset = FEATURES_FINAL).copy()
prod_df['y'] = prod_df['result'].map(y_map)

sc_final = StandardScaler()
X_final = sc_final.fit_transform(prod_df[FEATURES_FINAL])
m_final = LogisticRegression(max_iter = 2000)
m_final.fit(X_final, prod_df['y'], sample_weight = prod_df['recency_weight'])
print(f'Production model trained on {len(prod_df)} rows, all 20 teams, 2019-20 through 2025-26')

Production model trained on 4894 rows, all 20 teams, 2019-20 through 2025-26


In [23]:
season2627 = combined[combined['season'] == '2627'].copy()
next_fixture = season2627[season2627['scored'].isna()].sort_values(['team','gameweek']).groupby('team').first().reset_index()
SNAP_COLS = ['xG_roll5','xGA_roll5','pts_roll5','win_rate_roll5','xgd_roll5']
snapshot = next_fixture.set_index('team')[SNAP_COLS]

print('Frozen form snapshot, entering the next unplayed fixture, all 20 teams:')
print(snapshot.round(3).sort_values('xgd_roll5', ascending=False).head(8))
print()
print(f'Any missing snapshots: {snapshot.isna().any(axis=1).sum()}')

Frozen form snapshot, entering the next unplayed fixture, all 20 teams:
                   xG_roll5  xGA_roll5  pts_roll5  win_rate_roll5  xgd_roll5
team                                                                        
Arsenal               2.043      0.835       3.00            1.00      1.208
Manchester City       2.093      1.018       3.00            1.00      1.075
Brighton              2.695      1.847       1.75            0.50      0.849
Manchester United     2.115      1.375       1.00            0.25      0.740
Nottingham Forest     1.791      1.079       1.25            0.25      0.711
Bournemouth           1.788      1.426       0.75            0.00      0.363
Brentford             2.009      1.653       1.50            0.25      0.356
Liverpool             1.738      1.499       1.50            0.25      0.240

Any missing snapshots: 0


---
## Section H - Simulating One Match, Then One Season

Sample a discrete outcome from a probability triple, then assemble the remaining fixtures with each one's model perspective, tracked teams needing dynamic `stakes_intensity`, the rest fixed for the season.

In [24]:
def draw_outcome(p_loss, p_draw, u):
    """u is a uniform(0,1) draw. Returns 0=loss, 1=draw, 2=win."""
    if u < p_loss:
        return 0
    elif u < p_loss + p_draw:
        return 1
    else:
        return 2

# sanity check: does this reproduce the input probabilities over many draws?
p_loss, p_draw = 0.15, 0.25
draws = rng.random(10000)
outcomes = np.array([draw_outcome(p_loss, p_draw, u) for u in draws])
print('Target: L=0.15, D=0.25, W=0.60')
print('Observed:', pd.Series(outcomes).value_counts(normalize=True).sort_index().round(3).values)

Target: L=0.15, D=0.25, W=0.60
Observed: [0.148 0.248 0.604]


In [25]:
remaining = season2627[season2627['scored'].isna()].copy()
home_rows = remaining[remaining['venue']=='home'][['game_id','gameweek','date','team','opponent']].rename(
    columns={'team':'home_team','opponent':'away_team'})
fixtures = home_rows.sort_values(['gameweek','date']).reset_index(drop=True)
fixtures['perspective'] = fixtures['home_team']
fixtures['opponent'] = fixtures['away_team']

TEAMS20 = sorted(season2627['team'].unique())
T_IDX = {t: i for i, t in enumerate(TEAMS20)}

current_pts_row = season2627[season2627['scored'].notna()].sort_values(['team','gameweek']).groupby('team').last().reset_index()
current_pts = current_pts_row.set_index('team')['cum_pts'].reindex(TEAMS20).fillna(0).values.astype(float)

frozen_recent_form = {}
for team in TITLE_TEAMS:
    tdf = season2627[(season2627['team']==team) & (season2627['scored'].notna())].sort_values('gameweek')
    frozen_recent_form[team] = tdf['points'].tail(3).mean()

def is_rivalry_pair(team, opp):
    return opp in RIVALRIES.get(team, [])

static_rows = []
for _, r in fixtures.iterrows():
    own, opp = snapshot.loc[r['perspective']], snapshot.loc[r['opponent']]
    tracked = r['perspective'] in TITLE_TEAMS
    static_rows.append({
        'xG_roll5': own['xG_roll5'], 'xGA_roll5': own['xGA_roll5'],
        'pts_roll5': own['pts_roll5'], 'win_rate_roll5': own['win_rate_roll5'],
        'is_home': 1, 'is_big6_opp': int(r['opponent'] in BIG6),
        'opp_xgd_roll5': opp['xgd_roll5'], 'parity_gap': abs(own['xgd_roll5']-opp['xgd_roll5']),
        'is_non_big6_rivalry': int(tracked and is_rivalry_pair(r['perspective'], r['opponent']) and r['opponent'] not in BIG6),
        'form_mult': (1.15 if (tracked and frozen_recent_form[r['perspective']] < 1.0) else 1.0),
        'rivalry_bonus': (0.15 if (tracked and is_rivalry_pair(r['perspective'], r['opponent'])) else 0.0),
        'tracked': tracked,
    })
fixtures = pd.concat([fixtures, pd.DataFrame(static_rows)], axis=1)
fixtures['team_idx'] = fixtures['perspective'].map(T_IDX)

print(f'Remaining fixtures to simulate: {len(fixtures)} across {fixtures["gameweek"].nunique()} gameweeks')
print(f'Involving a tracked team: {fixtures["tracked"].sum()}, all others: {(~fixtures["tracked"]).sum()}')

Remaining fixtures to simulate: 340 across 34 gameweeks
Involving a tracked team: 68, all others: 272


In [26]:
def build_untracked_probs(fixtures, model, scaler, features):
    untracked = fixtures[~fixtures['tracked']].copy()
    feat = pd.DataFrame({
        'xG_roll5': untracked['xG_roll5'], 'xGA_roll5': untracked['xGA_roll5'],
        'pts_roll5': untracked['pts_roll5'], 'win_rate_roll5': untracked['win_rate_roll5'],
        'is_home': untracked['is_home'], 'is_big6_opp': untracked['is_big6_opp'],
        'opp_xgd_roll5': untracked['opp_xgd_roll5'], 'parity_gap': untracked['parity_gap'],
        'stakes_intensity': 0.0, 'is_non_big6_rivalry': 0, 'h2h_pts_avg3': 1.5,
    })[features]
    probs = model.predict_proba(scaler.transform(feat))
    untracked = untracked.reset_index(drop=True)
    untracked['pL'], untracked['pD'], untracked['pW'] = probs[:,0], probs[:,1], probs[:,2]
    return untracked

untracked_fx = build_untracked_probs(fixtures, m_final, sc_final, FEATURES_FINAL)
tracked_fx = fixtures[fixtures['tracked']].reset_index(drop=True)
print(f'Untracked fixtures with pre-computed probabilities: {len(untracked_fx)}')
print(f'Tracked fixtures needing per-run dynamic stakes: {len(tracked_fx)}')

Untracked fixtures with pre-computed probabilities: 272
Tracked fixtures needing per-run dynamic stakes: 68


---
## Section I - The Simulation Engine

Run the gameweek-by-gameweek season simulator across many runs, then check convergence, whether the run count actually settled on an answer, rather than assuming a round number is enough.

---
## Section J - Does the Simulator Actually Work?

Freeze at a past gameweek of a real, completed season and simulate forward, checking whether the actual final table falls inside the simulated distribution.

---
## Key Findings Summary